# TKCE — Colab Pro+ run (resumable)

**Setup:** `Runtime → Change runtime type → L4 GPU`, then run the cells top to bottom.

**Everything saves to Google Drive as it runs.** On **Pro+** it keeps running even if you
close the tab (up to 24 h). If it stops, re-run cells **3, 4, 6** and it continues where it
left off (nothing is lost).

**Tip:** progress lines appear ~1–2 minutes after starting the Full run (a dataset
downloads first).

## 1. Config

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"   # this project's public repo
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"         # everything is saved here

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print('Saving everything to:', DRIVE_BASE)

## 3. Clone the code + install (re-run after any disconnect)

In [ ]:
%cd /content
!rm -rf tkce_repo
!git clone $REPO_URL tkce_repo
%cd tkce_repo
!pip install -q -r requirements-tkce.txt
print('Repo + dependencies ready')

## 4. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L

## 5. (Optional) Time ONE dataset first — you can skip this

In [ ]:
print('Timing one dataset. First progress lines appear in ~1-2 minutes...')
!time python -u run_suite.py --tasks 361070 --seeds 0,1,2 --trials 100 --max-rows 16000 --device auto --out "$DRIVE_BASE/probe" 2>&1 | tee "$DRIVE_BASE/probe_run.log"

## 6. Full run — 15 datasets, resumable, saves to Drive
Re-run this cell to resume: it skips whatever already finished. Progress shows live.

In [ ]:
TASKS = "361070,361278,361062,361066,361280,361076,361072,361279,361286,361283,361110,361093,361094,361098,361287"
SUITE_OUT = DRIVE_BASE + "/suite_v1"
LOG = DRIVE_BASE + "/suite_v1_run.log"
print('Starting the full run. First progress lines appear in ~1-2 minutes.')
!python -u run_suite.py --tasks $TASKS --seeds 0,1,2 --trials 100 --max-rows 16000 --device auto --reference catboost --out "$SUITE_OUT" 2>&1 | tee -a "$LOG"

## 7. Check progress (use in a NEW session, not while cell 6 runs)

In [ ]:
!wc -l "$SUITE_OUT/results_long.csv" 2>/dev/null || echo "no results yet"
!echo '--- last log lines ---'
!tail -n 10 "$LOG" 2>/dev/null

### If it stops
Re-run **cell 3 → cell 4 → cell 6**. It continues from where it stopped.

## 8. Build the paper report (run any time, even on partial results)

In [ ]:
!python make_report.py --results "$SUITE_OUT/results_long.csv" --out "$SUITE_OUT/report"
!echo '--- average rank (lower = better) ---'
!cat "$SUITE_OUT/report/avg_rank.csv" 2>/dev/null

## 9. Analysis experiments (figures for the paper)

In [ ]:
FIG = DRIVE_BASE + "/figures"
AN  = DRIVE_BASE + "/analysis"
!python -u run_loss_ablation.py   --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"
!python -u run_mechanism.py       --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"
!python -u run_data_efficiency.py --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"
!python -u run_lambda_sweep.py    --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"

## 10. See everything that was saved

In [ ]:
!echo 'Files in your Drive results folder:'
!ls -R "$DRIVE_BASE" | head -80

In [ ]:
# ===== 11. DEEP-DIVE: training dynamics over MULTIPLE SHUFFLES =====
# Deeper Siamese encoder + deeper TabResNet, 600 epochs, slow LR, no early stopping.
# Repeats over 5 random train/val/test shuffles to check whether the test score is
# a genuine ceiling. Plots loss / AUC / accuracy (train vs val) + per-shuffle test AUC.
FIG = DRIVE_BASE + "/figures"
AN  = DRIVE_BASE + "/analysis"
!python -u run_deep_joint.py --task 361070 --seeds 0,1,2,3,4 --epochs 600 --lr 1e-4 --enc-width 512 --enc-depth 6 --embedding-dim 128 --d 256 --d-hidden 512 --n-blocks 8 --head tabresnet --device auto --out "$FIG" --csv "$AN"